# Shot Analyzer Demo

End-to-end walkthrough of the shot-analysis pipeline on a sample cricket clip.

**Pipeline stages**
1. Run existing YOLO ball detector + tracker over the clip
2. (One-time) calibrate the ground-plane homography by clicking 4 pitch corners
3. Run `analyse_delivery` to get the contact event, 2 s prediction, angle, and shot name
4. Render the annotated output video

> **Model note:** `models/weights/best.pt` detects two classes: **0 = ball**, **1 = bat**.
> Pass the same file for both `BALL_MODEL_PATH` and `BAT_MODEL_PATH`.

In [ ]:
import os, json
import numpy as np
import cv2

from shot_analyzer import (
    build_tracks_from_video,
    calibrate_from_clicks,
    save_homography,
    load_homography,
    analyse_delivery,
    render_overlay,
)

# ---- EDIT THESE PATHS ----
MODEL_PATH      = r"models/weights/best.pt"   # class 0 = ball, class 1 = bat
BALL_MODEL_PATH = MODEL_PATH
BAT_MODEL_PATH  = MODEL_PATH
VIDEO_PATH      = r"data/samples/test_video.mp4"
OUTPUT_VIDEO    = r"outputs/videos/shot_overlay.mp4"
HOMOGRAPHY_JSON = r"outputs/calibration_H.json"

HANDEDNESS = "right"  # or "left"

os.makedirs("outputs/videos", exist_ok=True)

## 1. Run detector + tracker

Uses the existing modules under `src/` â€” detection is not re-implemented.
The same `best.pt` model is used for both ball (class 0) and bat (class 1).

In [ ]:
ball_track, ball_boxes, bat_boxes, fps = build_tracks_from_video(
    VIDEO_PATH,
    ball_model_path=BALL_MODEL_PATH,
    bat_model_path=BAT_MODEL_PATH,
    ball_class_id=0,   # class 0 = ball
    bat_class_id=1,    # class 1 = bat
    ball_conf=0.2,
    bat_conf=0.25,
    include_ball_boxes=True,
)

print(f"fps={fps:.1f}")
print(f"ball detections : {len(ball_track)} frames")
print(f"ball boxes      : {len(ball_boxes)} frames")
print(f"bat  detections : {len(bat_boxes)} frames")

### No bat detections? Stand-in fallback

Skip this cell if `bat_boxes` is already populated.

In [ ]:
if not bat_boxes:
    cap = cv2.VideoCapture(VIDEO_PATH)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    cx, cy = w // 2, int(h * 0.72)
    bat_boxes = {f: (cx - 25, cy - 80, cx + 25, cy + 20) for f in range(n)}
    print(f"Stand-in bat_boxes at ({cx},{cy}) for {len(bat_boxes)} frames")
else:
    print("bat_boxes populated â€” skipping fallback.")

## 2. Calibrate the ground-plane homography (one-time per camera setup)

An OpenCV window opens on the first frame. Click 4 pitch corners in order:

1. Striker-end **off**-side corner
2. Striker-end **leg**-side corner
3. Bowler-end **leg**-side corner
4. Bowler-end **off**-side corner

Then press any key to confirm. `r` = reset clicks, `q` = abort.

If you already have `outputs/calibration_H.json`, use the *Load saved H* cell instead.

In [ ]:
H = calibrate_from_clicks(VIDEO_PATH, save_path=HOMOGRAPHY_JSON)
print("Homography saved to", HOMOGRAPHY_JSON)
print(H)

### Load saved H (skip calibration on repeat runs)

In [ ]:
# H = load_homography(HOMOGRAPHY_JSON)
# print(H)

## 3. Run the full analysis

In [ ]:
result = analyse_delivery(
    ball_track,
    bat_boxes,
    H,
    fps=fps,
    handedness=HANDEDNESS,
    proximity_px=30,
    angle_change_deg=25,
    contact_window=3,
    post_contact_frames=60,
    direction_lookahead=10,
)

print(json.dumps(result.to_dict(), indent=2)[:2000])

## 4. Render the annotated video

In [ ]:
if result.contact is not None:
    out = render_overlay(
        VIDEO_PATH, OUTPUT_VIDEO,
        ball_track, bat_boxes, result, H,
        ball_boxes=ball_boxes,
    )
    print("Overlay written to:", out)
else:
    print("No contact detected. Reason:", result.reason)

## 5. Summary

In [ ]:
if result.contact:
    print(f"Contact frame    : {result.contact.frame_idx}")
    print(f"Proximity        : {result.contact.proximity_px:.1f} px")
    print(f"Angle change     : {result.contact.angle_change_deg:.1f} deg")
    print(f"Shot angle       : {result.angle_deg:.1f} deg  (0=straight, 90=off, 270=leg)")
    print(f"Shot             : {result.shot_name}")
    print(f"Post-contact pts : {len(result.observed_ground)} observed, {len(result.predicted_ground)} predicted")
else:
    print("No contact:", result.reason)

## (Optional) Wagon-wheel plot

In [ ]:
try:
    import matplotlib.pyplot as plt

    if result.observed_ground or result.predicted_ground:
        fig, ax = plt.subplots(figsize=(6, 6))
        if result.observed_ground:
            xs = [p[1] for p in result.observed_ground]
            ys = [p[2] for p in result.observed_ground]
            ax.plot(xs, ys, 'o-', label='observed', markersize=4)
        if result.predicted_ground:
            xs = [p[1] for p in result.predicted_ground]
            ys = [p[2] for p in result.predicted_ground]
            ax.plot(xs, ys, '--', label='predicted (2 s)')
        ax.plot(0, 0, 'k^', markersize=12, label='striker')
        ax.axhline(20.12, color='gray', linestyle=':', label='bowler crease')
        ax.set_xlabel('X (m)  +X = off side (RH)')
        ax.set_ylabel('Y (m)  +Y = toward bowler')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.legend()
        ax.set_title(f"{result.shot_name}  ({result.angle_deg:.1f} deg)")
        plt.tight_layout()
        plt.show()
    else:
        print('No ground trajectory to plot.')
except ImportError:
    print('matplotlib not installed.')